In [0]:
dbutils.widgets.removeAll()

In [0]:
# Leer resultado persistente de Ingest

df_conceptos_ingest = spark.table(
    "catalog_au.bronze.sfc_conceptos_ingest"
)

print(
    "Registros recibidos desde Bronze:",
    df_conceptos_ingest.count()
)

In [0]:
from pyspark.sql import functions as F

df_conceptos_normalizados = (
    df_conceptos_ingest
    .withColumn(
        "nombre_normalizado",
        F.upper(
            F.regexp_replace(
                F.trim(F.col("nombre_concepto_raw")),
                r"\s+",
                " "
            )
        )
    )
    .groupBy(
        "codigo_concepto",
        "fila_excel",
        "nivel_excel",
        "nombre_normalizado"
    )
    .agg(
        F.min("anio_origen").alias("primer_anio"),
        F.max("anio_origen").alias("ultimo_anio"),
        F.countDistinct("anio_origen").alias("anios_presentes")
    )
)

print(
    "Conceptos normalizados:",
    df_conceptos_normalizados.count()
)

display(
    df_conceptos_normalizados
    .orderBy("fila_excel")
    .limit(20)
)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

ventana = (
    Window
    .orderBy("fila_excel")
    .rowsBetween(Window.unboundedPreceding, 0)
)

df_conceptos_jerarquia = (
    df_conceptos_normalizados
    .withColumn(
        "padre_nivel_1",
        F.last(
            F.when(
                F.col("nivel_excel") == 1,
                F.col("codigo_concepto")
            ),
            ignorenulls=True
        ).over(ventana)
    )
    .withColumn(
        "ultimo_nivel_2",
        F.last(
            F.when(
                F.col("nivel_excel") == 2,
                F.col("codigo_concepto")
            ),
            ignorenulls=True
        ).over(ventana)
    )
    .withColumn(
        "codigo_padre",
        F.when(
            F.col("nivel_excel") == 1,
            F.lit(None).cast("string")
        )
        .when(
            F.col("nivel_excel") == 2,
            F.col("padre_nivel_1")
        )
        .otherwise(F.col("ultimo_nivel_2"))
    )
    .drop("padre_nivel_1", "ultimo_nivel_2")
)

display(
    df_conceptos_jerarquia
    .select(
        "codigo_concepto",
        "nombre_normalizado",
        "nivel_excel",
        "codigo_padre"
    )
    .orderBy("fila_excel")
    .limit(20)
)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Orden de los conceptos
w_orden = (
    Window
    .orderBy("fila_excel")
    .rowsBetween(Window.unboundedPreceding, 0)
)

# Identificar el último concepto de nivel 1
df_jerarquia_base = (
    df_conceptos_normalizados
    .withColumn(
        "padre_nivel_1",
        F.last(
            F.when(
                F.col("nivel_excel") == 1,
                F.col("codigo_concepto")
            ),
            ignorenulls=True
        ).over(w_orden)
    )
)

# Reiniciar el seguimiento de nivel 2 por cada sección
w_nivel_2 = (
    Window
    .partitionBy("padre_nivel_1")
    .orderBy("fila_excel")
    .rowsBetween(Window.unboundedPreceding, 0)
)

df_conceptos_jerarquia = (
    df_jerarquia_base
    .withColumn(
        "padre_nivel_2",
        F.last(
            F.when(
                F.col("nivel_excel") == 2,
                F.col("codigo_concepto")
            ),
            ignorenulls=True
        ).over(w_nivel_2)
    )
    .withColumn(
        "codigo_padre",
        F.when(
            F.col("nivel_excel") == 1,
            F.lit(None).cast("string")
        )
        .when(
            F.col("nivel_excel") == 2,
            F.col("padre_nivel_1")
        )
        .otherwise(F.col("padre_nivel_2"))
    )
    .drop("padre_nivel_1", "padre_nivel_2")
)

print(
    "Conceptos con jerarquía:",
    df_conceptos_jerarquia.count()
)

In [0]:
# Conceptos sin padre que deberían tenerlo
df_sin_padre = df_conceptos_jerarquia.filter(
    (F.col("nivel_excel") > 1) &
    F.col("codigo_padre").isNull()
)

# Referencias a padres inexistentes
df_padres_invalidos = (
    df_conceptos_jerarquia
    .filter(F.col("codigo_padre").isNotNull())
    .join(
        df_conceptos_jerarquia
        .select(
            F.col("codigo_concepto").alias("codigo_padre")
        ),
        on="codigo_padre",
        how="left_anti"
    )
)

print("Conceptos totales:", df_conceptos_jerarquia.count())
print("Conceptos sin padre:", df_sin_padre.count())
print("Referencias a padres inexistentes:", df_padres_invalidos.count())

display(df_sin_padre)

In [0]:
from pyspark.sql import functions as F

df_conceptos_jerarquia = (
    df_conceptos_jerarquia
    .withColumn(
        "codigo_padre",
        F.when(
            (F.col("nivel_excel") == 3) &
            F.col("codigo_padre").isNull(),
            F.last(
                F.when(
                    F.col("nivel_excel") == 1,
                    F.col("codigo_concepto")
                ),
                ignorenulls=True
            ).over(w_orden)
        ).otherwise(F.col("codigo_padre"))
    )
)

df_sin_padre = df_conceptos_jerarquia.filter(
    (F.col("nivel_excel") > 1) &
    F.col("codigo_padre").isNull()
)

print("Conceptos totales:", df_conceptos_jerarquia.count())
print("Conceptos sin padre:", df_sin_padre.count())

In [0]:
from pyspark.sql import functions as F

df_hijos = df_conceptos_jerarquia.alias("h")

df_padres = (
    df_conceptos_jerarquia
    .select(
        F.col("codigo_concepto").alias("codigo_padre_ref"),
        F.col("nivel_excel").alias("nivel_padre"),
        F.col("fila_excel").alias("fila_padre")
    )
    .alias("p")
)

df_validacion_jerarquia = (
    df_hijos
    .join(
        df_padres,
        F.col("h.codigo_padre") == F.col("p.codigo_padre_ref"),
        "left"
    )
    .select(
        F.col("h.codigo_concepto"),
        F.col("h.nombre_normalizado"),
        F.col("h.nivel_excel"),
        F.col("h.fila_excel"),
        F.col("h.codigo_padre"),
        F.col("p.nivel_padre"),
        F.col("p.fila_padre")
    )
)

df_errores_jerarquia = df_validacion_jerarquia.filter(
    (
        F.col("nivel_excel") == 1
    ) & F.col("codigo_padre").isNotNull()
    |
    (
        (F.col("nivel_excel") > 1) &
        (
            F.col("codigo_padre").isNull() |
            F.col("nivel_padre").isNull() |
            (F.col("nivel_padre") >= F.col("nivel_excel")) |
            (F.col("fila_padre") >= F.col("fila_excel"))
        )
    )
)

print("Conceptos analizados:", df_validacion_jerarquia.count())
print("Relaciones inválidas:", df_errores_jerarquia.count())

display(df_errores_jerarquia)

In [0]:
display(
    df_validacion_jerarquia
    .filter(
        F.col("fila_excel").between(539, 555)
    )
    .orderBy("fila_excel")
)

In [0]:
print(
    "Conceptos analizados:",
    df_validacion_jerarquia.count()
)

print(
    "Relaciones inválidas:",
    df_errores_jerarquia.count()
)

print(
    "Conceptos únicos:",
    df_conceptos_jerarquia
    .select("codigo_concepto")
    .distinct()
    .count()
)

In [0]:
%sql
DESCRIBE TABLE catalog_au.golden.dim_concepto_financiero;

In [0]:
%sql
SELECT COUNT(*) AS total_registros
FROM catalog_au.golden.dim_concepto_financiero;

In [0]:
from pyspark.sql import functions as F

df_codigos_padres = (
    df_conceptos_jerarquia
    .filter(F.col("codigo_padre").isNotNull())
    .select(
        F.col("codigo_padre").alias("codigo_concepto"),
        F.lit(True).alias("tiene_hijos")
    )
    .distinct()
)

df_conceptos_hojas = (
    df_conceptos_jerarquia
    .join(
        df_codigos_padres,
        on="codigo_concepto",
        how="left"
    )
    .withColumn(
        "es_concepto_hoja",
        F.col("tiene_hijos").isNull()
    )
    .drop("tiene_hijos")
)

display(
    df_conceptos_hojas
    .groupBy("nivel_excel", "es_concepto_hoja")
    .count()
    .orderBy("nivel_excel", "es_concepto_hoja")
)

In [0]:
df_base = df_conceptos_hojas.alias("c")

df_padre = df_conceptos_hojas.select(
    F.col("codigo_concepto").alias("p_codigo"),
    F.col("codigo_padre").alias("p_padre"),
    F.col("nombre_normalizado").alias("p_nombre")
).alias("p")

df_abuelo = df_conceptos_hojas.select(
    F.col("codigo_concepto").alias("a_codigo"),
    F.col("nombre_normalizado").alias("a_nombre")
).alias("a")

df_conceptos_rutas = (
    df_base
    .join(
        df_padre,
        F.col("c.codigo_padre") == F.col("p.p_codigo"),
        "left"
    )
    .join(
        df_abuelo,
        F.col("p.p_padre") == F.col("a.a_codigo"),
        "left"
    )
    .withColumn(
        "ruta_jerarquia",
        F.when(
            F.col("c.codigo_padre").isNull(),
            F.col("c.codigo_concepto")
        )
        .when(
            F.col("p.p_padre").isNull(),
            F.concat_ws(
                "/",
                F.col("p.p_codigo"),
                F.col("c.codigo_concepto")
            )
        )
        .otherwise(
            F.concat_ws(
                "/",
                F.col("a.a_codigo"),
                F.col("p.p_codigo"),
                F.col("c.codigo_concepto")
            )
        )
    )
    .withColumn(
        "categoria_financiera",
        F.coalesce(
            F.col("a.a_nombre"),
            F.col("p.p_nombre"),
            F.col("c.nombre_normalizado")
        )
    )
    .select(
        F.col("c.codigo_concepto"),
        F.col("c.nombre_normalizado").alias("nombre_concepto"),
        F.col("c.codigo_padre"),
        F.col("c.nivel_excel").alias("nivel_jerarquia"),
        "ruta_jerarquia",
        "categoria_financiera",
        F.col("c.es_concepto_hoja")
    )
)

print("Conceptos preparados:", df_conceptos_rutas.count())

display(
    df_conceptos_rutas
    .orderBy("codigo_concepto")
    .limit(20)
)

In [0]:
from pyspark.sql import functions as F

df_conceptos_transform = (
    df_conceptos_rutas
    .select(
        F.col("codigo_concepto"),
        F.col("nombre_concepto"),
        F.col("codigo_padre"),
        F.col("nivel_jerarquia").cast("int"),
        F.col("ruta_jerarquia"),
        F.col("categoria_financiera"),
        F.lit(None).cast("string").alias("unidad_medida"),
        F.col("es_concepto_hoja").cast("boolean"),
        F.lit("ACTIVO").alias("estado_concepto")
    )
)

df_conceptos_transform.createOrReplaceTempView(
    "vw_sfc_conceptos_transform"
)

print(
    "Conceptos listos para Load:",
    df_conceptos_transform.count()
)

df_conceptos_transform.printSchema()

In [0]:
total = df_conceptos_transform.count()

codigos_unicos = (
    df_conceptos_transform
    .select("codigo_concepto")
    .distinct()
    .count()
)

nulos_obligatorios = (
    df_conceptos_transform
    .filter(
        F.col("codigo_concepto").isNull() |
        F.col("nombre_concepto").isNull() |
        F.col("ruta_jerarquia").isNull() |
        F.col("categoria_financiera").isNull()
    )
    .count()
)

print("Total conceptos:", total)
print("Códigos únicos:", codigos_unicos)
print("Duplicados:", total - codigos_unicos)
print("Nulos obligatorios:", nulos_obligatorios)

assert total == 589
assert total == codigos_unicos
assert nulos_obligatorios == 0

print("VALIDACIÓN TRANSFORM: OK")

In [0]:
# Publicar conceptos transformados en Silver

tabla_destino = "catalog_au.silver.sfc_conceptos_transform"

(
    df_conceptos_transform
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(tabla_destino)
)

print("Tabla publicada:", tabla_destino)

print(
    "Conceptos disponibles para Load:",
    spark.table(tabla_destino).count()
)